In [ ]:
import torch
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader import *
# Historical configuration omitted; provide values through private configuration.
# from loss_IMP import *

from GPN_V3_IMP_onlyTrainer import GPNTrainerWithIMP
from GPN_V4 import GPN_Optimized,count_parameters
import higher
# from draw_pic import *#原文件已重命名
from thop import profile, clever_format
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
LOSS_FEATURE_DIM = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.LOSS_FEATURE_DIM')
N_WAY = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.EPOCHS')
BLR = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.BLR')
LOSSLR=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.LOSSLR')
MLR = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.MLR')
BATCH_SIZE =_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.BATCH_SIZE')
loss_alpha=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.loss_alpha')
SEED = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.SEED')
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.path__2')
# 数据路径
# current_path=Path(__file__).resolve().parent
dataset_path = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.module.dataset_path')
    
def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.use_wavelet_denoise'), 
                             wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.wavelet_name'), 
                             denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.denoise_level'))
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.max_sample_count__2'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.use_wavelet_denoise__2'), 
                             wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.wavelet_name__2'), 
                             denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.denoise_level__2'))
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    
    # 创建元任务划分集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model = GPN(use_se=True)#GPN_change
    # model = GPN_Optimized(use_repvgg=True, use_convnext=True)#GPN_new
    model = GPN_Optimized(use_repvgg=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.use_repvgg'),
                 use_convnext=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.use_convnext'),
                 attention_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.attention_type'),#se,decoupled,freq_priority
                 reduction=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.reduction'))

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.size_or_budget'),_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.size_or_budget__3'), _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    # meta_learner = GPNTrainer(model,device=device,
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    meta_learner = GPNTrainerWithIMP(
        crop_ratio_h=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.crop_ratio_l'), resize=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.resize'),
                    model=model,
                    device=device,
                    lr=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.lr'),
                    alpha=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.alpha'),
                    sigma_init=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.sigma_init'),
                    prototype_mode=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.prototype_mode'),
                    distance_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.main.distance_type')
                )
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")

    # meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.data_dir_path'),seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.use_wavelet_denoise'), 
                            wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.wavelet_name'), 
                            denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.denoise_level'))
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.num_tasks'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.batch_size'), shuffle=True)

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # model = GPN(use_se=True)
    model = GPN_Optimized(use_repvgg=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.use_repvgg'),
                 use_convnext=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.use_convnext'),
                 attention_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.attention_type'),#se,decoupled,freq_priority
                 reduction=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.reduction'),
                 freq_time_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.freq_time_ratio'))
    # 创建trainer并加载模型
    # trainer = GPNTrainer(model, device=device)
    
    trainer = GPNTrainerWithIMP(crop_ratio_h=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.crop_ratio_l'), resize=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.resize'),
                    model=model,
                    device=device,
                    alpha=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.alpha'),
                    sigma_init=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.sigma_init'),
                    prototype_mode=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.prototype_mode'),
                    distance_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.distance_type')
                )
    trainer.load_model(_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell1.inference.path'))
    
    # 评估
    result_pack=trainer.evaluate(test_loader=test_loader,n_ways=N_WAY,all_result=True)
    print('avg_time_ms',result_pack['avg_time_ms'])
    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()


In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#Kmeans++聚类
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm

from GPN_V3 import *
import higher
# from draw_pic import *
from thop import profile, clever_format
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
N_WAY = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.EPOCHS')
BLR = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.BLR')
LOSSLR=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.LOSSLR')
MLR = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.MLR')
BATCH_SIZE =_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.SEED')
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.LOAD_MODEL_PATH')
# 数据路径
# current_path=Path(__file__).resolve().parent
# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.use_wavelet_denoise'), 
                             wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.wavelet_name'), 
                             denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.denoise_level'),)
                             # Historical configuration omitted; provide values through private configuration.
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.max_sample_count__2'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.use_wavelet_denoise__2'), 
                             wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.wavelet_name__2'), 
                             denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.denoise_level__2'),)
                             # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    
    # 创建元任务划分集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model = GPN(use_se=True)#GPN_change
    # model = GPN_Optimized(use_repvgg=True, use_convnext=True)#GPN_new
    model = GPN_Optimized(use_repvgg=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.use_repvgg'),
                 use_convnext=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.use_convnext'),
                 attention_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.attention_type'),
                 reduction=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.reduction'),
                 freq_time_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.freq_time_ratio'))

    params = count_parameters(model)
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.size_or_budget'),_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.size_or_budget__3'), _cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_multi=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.use_multi'),
                              crop_ratio_h=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.crop_ratio_l'),resize=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.resize'),
                              prototypes_per_class=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.main.prototypes_per_class'))
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")

    # meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.data_dir_path'),seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.use_wavelet_denoise'), 
                            wavelet_name=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.wavelet_name'), 
                            denoise_level=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.denoise_level'),)
                            # Historical configuration omitted; provide values through private configuration.

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # model = GPN(use_se=True)
    model = GPN_Optimized(use_repvgg=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.use_repvgg'),
                 use_convnext=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.use_convnext'),
                 attention_type=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.attention_type'),
                 reduction=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.reduction'),
                 freq_time_ratio=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.freq_time_ratio'))
    # 创建trainer并加载模型
    # trainer = GPNTrainer(model, device=device)
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_multi=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.use_multi'),
                              crop_ratio_h=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.crop_ratio_l'),resize=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.resize'),
                              prototypes_per_class=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.prototypes_per_class'))
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.num_tasks'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.batch_size'), shuffle=True)

    # # 调用可视化
    # print("\nStarting t-SNE visualization...")
    # # 使用 test_loader 进行特征提取
    # meta_learner.visualize_tsne(test_loader, 
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # 评估
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.n_trials'),q_query=Q_QUERY)
    plot_all_visualizations(results,output_dir=_cfg_require('GPN_V3_IMP_and_Kmeans++.ipynb.cell2.inference.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5
    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()
